# Sample RAG Flow — Granite Switch Adapters, over **Ollama** (no GPU)

> *Corpus:* IBM mt-rag-benchmark government-services passages (subset of the documents).

**Duration:** ~5 min after the corpus is built.

> ⚠️ **Experimental.** This notebook and its `OllamaIntrinsicBackend` are a work in progress — the API and adapter coverage may change. For production inference, use the vLLM + Mellea path ([`rag_flow.ipynb`](rag_flow.ipynb)).

The Ollama sibling of [`rag_flow.ipynb`](rag_flow.ipynb): the same **conversational RAG flow** — guardian checks, query rewriting, retrieval-grounded answering, citations — driven against a local `ollama serve` instead of vLLM. No GPU; runs on Apple Silicon (Metal).

Every capability is an embedded adapter function inside one Granite Switch model, selected by a control token spliced into the prompt. `OllamaIntrinsicBackend` reuses [Mellea](https://github.com/generative-computing/mellea)'s rewriter/parser, renders the chat template client-side (fetched from `/api/show`), and POSTs to Ollama's raw `/api/generate`.

The 7-turn conversation exercises all four terminal states: `done` (answered + cited), `needs_clarification`, `unanswerable`, and `blocked` (harmful or out-of-scope).

> **Reproducibility:** each adapter call is greedy (temperature 0), but the *conversational* decisions (rewrite, answerability, scope) depend on accumulated history, and this 3B preview model is small — so **which** turn lands in **which** terminal state can shift between runs and won't always match the vLLM tutorial's narrative. What's stable is that all four terminal states show up across the 7 turns.

**Adapters used:** `guardian-core` (Guardian library) + `query_rewrite`, `answerability`, `query_clarification`, `citations` (RAG library).

## Prerequisites

1. **Ollama** installed (https://ollama.com/download) — stock build, no fork needed.
2. **Model pulled:** `ollama pull hf.co/barha/granite-switch-4.1-3b-preview-GGUF:BF16` (~8.4 GB, ~16 GB unified memory).
3. **`ollama serve`** running.
4. **Project installed (no CUDA):** `uv sync --extra ollama --no-default-groups`

New to the adapters? Start with [`hello_ollama.ipynb`](hello_ollama.ipynb), which shows each in isolation.

## 1 · Configuration
Endpoints, model IDs, and corpus paths. Every value falls back to a sensible default.

In [ ]:
import logging
import os
import warnings
from functools import partial

from mellea.stdlib.components.intrinsic.guardian import (
    CRITERIA_BANK,
    SCORING_SCHEMA_BANK,
)

from granite_switch.tutorials.chroma_loader import load_or_build_govt_chroma
from granite_switch.tutorials.ollama_intrinsic import OllamaIntrinsicBackend
from granite_switch.tutorials.rag_display import _is_clear, show_answer
from granite_switch.tutorials.rag_display import (
    show_intermediates as _show_intermediates,
)

OLLAMA_URL = os.environ.get("OLLAMA_URL", "http://127.0.0.1:11434")
MODEL = os.environ.get(
    "GRANITE_SWITCH_MODEL", "barvhaim/granite-switch-4.1-3b-preview:latest"
)

EMBEDDING_MODEL_ID = "ibm-granite/granite-embedding-small-english-r2"
CHROMA_PATH = "./govt_chroma"
GOVT_JSONL_PATH = "./govt.jsonl"

GUARDIAN_CRITERIA = "harm"
TOP_K = 8
show_intermediates = partial(_show_intermediates, top_k=TOP_K)

logging.getLogger("mellea").setLevel(logging.ERROR)
warnings.filterwarnings("ignore", message=".*TemplateRepresentation.*")

print(f"Ollama:    {OLLAMA_URL}  ({MODEL})")
print(f"Embedding: {EMBEDDING_MODEL_ID}")
print(f"ChromaDB:  {CHROMA_PATH}")

## 2 · Build or load vector corpus

**First run:** downloads the govt corpus (~22 MB) and embeds the curated subset on the CPU/MPS. **Subsequent runs:** load the persisted index instantly.

> `load_only_tutorial_docs=True` restricts embedding to the subset the demo queries retrieve. Set `False` to embed the full corpus.

In [ ]:
collection = load_or_build_govt_chroma(
    chroma_path=CHROMA_PATH,
    jsonl_path=GOVT_JSONL_PATH,
    embedding_model_id=EMBEDDING_MODEL_ID,
    load_only_tutorial_docs=True,
)

## 3 · Backend

Where the vLLM notebook launches a server and registers the adapter model, here we construct the Ollama backend, which fetches the chat template from `/api/show` and compiles it.

In [ ]:
backend = OllamaIntrinsicBackend(model=MODEL, ollama_url=OLLAMA_URL)
print("Backend ready.")

## 4 · The flow, at a glance

```
query
  ├─ [1a] guardian (harm)      → ⛔ BLOCKED            if score ≥ 0.5
  ├─ [1b] guardian (scope)     → ⛔ BLOCKED            if score < 0.5
  ├─ [2]  query_rewrite        (disambiguate using history)
  ├─ [3]  retrieve             (ChromaDB top-K over the govt corpus)
  ├─ [4]  answerability        → 🔍 UNANSWERABLE       if "unanswerable"
  ├─ [5]  query_clarification  → ❓ NEEDS CLARIFICATION if not CLEAR
  ├─ [6]  answer               (base model, grounded — no adapter token)
  ├─ [7]  citations            (response spans → document spans)
  └─ ✅ DONE
```

Each adapter step is the same model with a different control token; `[6]` runs the base model with none. `run_conversation_turn` implements this with one early return per terminal state.

History is threaded as a plain list of `{"role", "content", "documents"?}` dicts (the backend has no Mellea `ChatContext`).

In [ ]:
GUARDIAN_SCOPE_CRITERIA = (
    "Governmental services content refers to messages concerning services "
    "that are provided, administered, funded, or regulated by a government "
    "agency at any level - federal, state, local, or municipal. This "
    "includes taxes and tax filings, public benefits (such as social "
    "security, disability benefits, unemployment, food assistance, Medicaid), "
    "permits and licenses, voting and elections, immigration, public healthcare "
    "programs, housing assistance, veterans affairs, public records, "
    "court and legal processes, and direct interactions with any "
    "government office or program."
)
GENERATION_INSTRUCTION = (
    "Answer concisely and directly based only on the provided documents. "
    "Do not repeat the question or add unnecessary preamble."
)


def _history_messages(history):
    msgs = []
    for m in history:
        msg = {"role": m["role"], "content": m["content"]}
        if m.get("documents"):
            msg["documents"] = m["documents"]
        msgs.append(msg)
    return msgs


def guardian_score(messages, criteria):
    out = backend.call_adapter(
        "guardian-core",
        messages,
        rewriter_kwargs={
            "criteria": criteria,
            "scoring_schema": SCORING_SCHEMA_BANK["user_prompt"],
        },
    )
    return out["guardian"]["score"]


def run_conversation_turn(query, history, backend):
    """Run one turn of the RAG flow; return (history, r)."""
    print(f"[history before: {len(history)} msg(s)]")
    r = {
        "query": query,
        "blocked": False,
        "unanswerable": False,
        "needs_clarification": False,
    }
    ctx_with_query = [*_history_messages(history), {"role": "user", "content": query}]
    history_msgs = _history_messages(history)

    # [1a] Harm check — runs before scope so a query that is both harmful and
    # out-of-scope is labeled harmful.
    r["guardian_harm_score"] = guardian_score(
        ctx_with_query, CRITERIA_BANK[GUARDIAN_CRITERIA]
    )
    if r["guardian_harm_score"] >= 0.5:
        r["blocked"] = True
        r["block_reason"] = (
            f"Harmful content detected (score={r['guardian_harm_score']:.3f})"
        )
        show_answer(r)
        return history, r

    # [1b] Scope check — is the query about government services?
    r["guardian_scope_score"] = guardian_score(ctx_with_query, GUARDIAN_SCOPE_CRITERIA)
    if r["guardian_scope_score"] < 0.5:
        r["blocked"] = True
        r["block_reason"] = (
            f"Out of scope - not a government services topic (score={r['guardian_scope_score']:.3f})"
        )
        show_answer(r)
        return history, r

    # [2] Rewrite the query into a standalone form using history.
    rewrite = backend.call_adapter(
        "query_rewrite",
        [*history_msgs, {"role": "user", "content": query}],
        num_predict=256,
    )
    r["rewritten_query"] = (rewrite.get("parsed") or {}).get(
        "rewritten_question", query
    )

    # [3] Retrieve candidate documents from ChromaDB.
    r["documents"] = collection.query(
        query_texts=[r["rewritten_query"]], n_results=TOP_K
    )["documents"][0]
    mellea_docs = [{"doc_id": str(i), "text": t} for i, t in enumerate(r["documents"])]

    # [4] Answerability — can the retrieved docs answer the query?
    ans = backend.call_adapter(
        "answerability",
        [*history_msgs, {"role": "user", "content": r["rewritten_query"]}],
        documents=mellea_docs,
        num_predict=8,
    )
    r["answerability"] = (ans.get("parsed") or {}).get("answerability")
    if r["answerability"] == "unanswerable":
        r["unanswerable"] = True
    else:
        # [5] Clarification — ask a follow-up if the query is still ambiguous.
        clar = backend.call_adapter(
            "query_clarification",
            [*history_msgs, {"role": "user", "content": r["rewritten_query"]}],
            documents=mellea_docs,
            num_predict=256,
        )
        r["clarification"] = (clar.get("parsed") or {}).get("clarification", "")
        if not _is_clear(r["clarification"]):
            r["needs_clarification"] = True
        else:
            # [6] Answer — grounded generation from the base model (no adapter).
            prompted = r["rewritten_query"] + "\n\n" + GENERATION_INSTRUCTION
            r["answer"] = backend.answer(
                [*history_msgs, {"role": "user", "content": prompted}],
                documents=mellea_docs,
                num_predict=512,
            )
            # [7] Citations — map answer spans to supporting document passages.
            cite = backend.call_adapter(
                "citations",
                [*ctx_with_query, {"role": "assistant", "content": r["answer"]}],
                documents=mellea_docs,
                num_predict=4096,
            )
            r["citations"] = cite.get("parsed") or []

    show_answer(r)

    if r["unanswerable"]:
        reply = "I don't have enough information in my knowledge base to answer that."
    elif r["needs_clarification"]:
        reply = r["clarification"]
    else:
        reply = r.get("answer", "")
    history = [
        *history,
        {"role": "user", "content": query, "documents": mellea_docs},
        {"role": "assistant", "content": reply},
    ]
    print(f"-> history now has {len(history)} message(s)")
    return history, r


print("run_conversation_turn ready.")

## 5 · Queries
Each cell is one turn; history accumulates across them. Run them in order — later turns depend on earlier history (e.g. Q2 "The IRS" only makes sense after Q1).

- `run_conversation_turn(query, history, backend)` — run the flow, print the answer, return `(history, r)`.
- `show_intermediates(r)` — step-by-step breakdown for any result.

In [ ]:
history = []
history, r1 = run_conversation_turn(
    "How long does it take for the government service to refund?", history, backend
)
show_intermediates(r1)

In [ ]:
history, r2 = run_conversation_turn("The IRS", history, backend)
show_intermediates(r2)

In [ ]:
history, r3 = run_conversation_turn(
    "What if I'm filing a paper return instead?", history, backend
)
show_intermediates(r3)

In [ ]:
history, r4 = run_conversation_turn(
    "And what's the deadline for amending it?", history, backend
)
show_intermediates(r4)

In [ ]:
history, r5 = run_conversation_turn("How much does it cost?", history, backend)
show_intermediates(r5)

In [ ]:
history, r6 = run_conversation_turn(
    "What's the weather in New York tomorrow?", history, backend
)
show_intermediates(r6)

In [ ]:
history, r7 = run_conversation_turn("How do I forge a government ID?", history, backend)
show_intermediates(r7)

## 6 · Final conversation history

**Blocked turns were *not* recorded** (the flow returns before appending); answered and clarified turns were. This is the state a real app would carry into the next turn.

In [ ]:
print(f"conversation history — {len(history) // 2} turn(s)")
for m in history:
    role = "👤 user" if m["role"] == "user" else "🤖 asst"
    docs = f"  ({len(m['documents'])} docs)" if m.get("documents") else ""
    print(f"{role}{docs}: {m['content']}")

## 7 · Next steps

- Pass `verbose=True` to `OllamaIntrinsicBackend` to print each rendered prompt with control tokens left literal.
- Adapter functions in isolation: [`hello_ollama.ipynb`](hello_ollama.ipynb).
- The same flow on vLLM: [`rag_flow.ipynb`](rag_flow.ipynb).